# Domain Randomization vs. Segmented Learning Flow

Evaluates the hypothesis **segmented learning beats domain randomization** across matched visual proprioception models (`Conv-VAE-Neo`, `ProprioTuned ViT`, and optionally `VGG19` / `ResNet50`) using YAML-configured preprocessing pipelines.

Conditions compared per architecture (adjacent bars in `*23` layout):
- `orig` — trained raw, tested raw / cluttered
- `seg_both` — trained on `seg_white`, tested on `seg_white` (live `SmallSegUNet` re-segmentation on Suite B1 cluttered frames)
- `seg_train_real_test` — **same weights as `seg_both`**, tested directly on raw / cluttered frames (zero segmentation at inference)
- `dr_photo` — trained with mask-free photometric/lighting/noise DR, tested raw / cluttered
- `dr_bg_swap` — trained with mask-based random synthetic backgrounds, tested raw / cluttered
- `dr_full` — trained with random backgrounds + distractors + lighting + color jitter + noise + cutout, tested raw / cluttered

Two strictly held-out evaluation suites on unseen `vp_testing` (episodes `8..9`):
- **Suite A (`vp_comp_clean`):** clean real `vp_testing` frames
- **Suite B1 (`vp_comp_ood`):** disjoint unseen clutter + lighting stress test (`eval_ood_clutter.yaml`)

In [ ]:
import copy
import pathlib
import pprint
import shutil
import sys
import yaml

_cwd = pathlib.Path.cwd().resolve()
for _candidate in (_cwd / "src", _cwd, _cwd / "..", _cwd / "../.."):
    _cand = _candidate.resolve()
    if (_cand / "exp_run_config.py").is_file():
        SRC_ROOT = _cand
        break
else:
    raise RuntimeError(f"Could not locate BerryPicker/src from {_cwd}")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import numpy as np
import torch
from IPython.display import Image, Markdown, display

from flow import setup_flow, flow_entry, run_flow, display_flow_report
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard
from _experimental._2026_11_robot_segmentation import load_segmenter
from _experimental._2026_11_domain_randomization import DomainRandomizer
from _experimental._2026_11_domain_randomization.dataset_builder import (
    cache_episode_masks,
    make_techniques_contact_sheet,
    materialize_randomized_demopack,
    read_video_rgb,
    summarize_dr_comparisons,
)

RECIPES_DIR = SRC_ROOT / "_experimental" / "_2026_11_domain_randomization" / "recipes"
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| runtime device:", Config().runtime["device"])

## Flow Parameters

Defaults target a fast laptop smoke run (`szenes`, Apple M3 Max MPS). Override via `papermill` for a full sweep on `glassy`.

In [ ]:
flow_name = "DomainRand_smoke_01"
demopack_name = "random-both-cameras-video"
demonstration_cam = "dev2"
seg_model_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot/models/unet_sam3_dev2/best"
seg_device = "auto"
conditions = ["orig", "seg_both", "seg_train_real_test", "dr_bg_swap", "dr_full"]
copies_per_episode = 1
latent_sizes = [128]
do_VAE_NEO = True
do_VIT = True
do_VGG = False
do_RESNET = False
vae_image_size = [128, 128]
vae_frame_stride = 2
epochs_sp_vae = 5
epochs_sp_vit = 3
epochs_sp_cnn = 3
epochs_vp = 20
image_size = [256, 256]
creation_style = "exist-ok"
compare_creation_style = "discard-old"
overwrite_dr_demos = False

## Stage 0: Cache Robot Masks, Render Contact Sheet, and Materialize DR & OOD Suites

In [ ]:
expruns_path, results_path, notebooks_path = setup_flow(flow_name, [
    "robot_al5d",
    "demonstration",
    "sensorprocessing_conv_vae_neo",
    "sensorprocessing_propriotuned_cnn",
    "sensorprocessing_propriotuned_Vit",
    "visual_proprioception",
    "visual_proprioception_collections",
])

demopack_path = pathlib.Path(Config()["demopacks_path"], demopack_name).expanduser()
if not demopack_path.exists():
    raise FileNotFoundError(f"Demopack not found: {demopack_path}")

selection = import_demopack(demopack_path, group_chooser_sp_vp_standard)
pprint.pprint(selection)

all_episodes = sorted({
    ep
    for split_key in ("sp_training", "sp_validation", "vp_training", "vp_validation", "vp_testing")
    for ep in selection[split_key]
})
train_episodes = sorted(set(selection["sp_training"]) | set(selection["vp_training"]))
test_episodes = list(selection["vp_testing"])

exp_src_demo = Config().get_experiment("demonstration", demopack_name)
src_yaml_path = pathlib.Path(Config().get_exprun_path(), "demonstration", f"{demopack_name}.yaml")
src_demo_root = pathlib.Path(exp_src_demo.data_dir())

seg_model_path = pathlib.Path(seg_model_dir).expanduser()
if not seg_model_path.exists():
    raise FileNotFoundError(f"Robot segmenter checkpoint not found: {seg_model_path}")
segmenter = load_segmenter(seg_model_path, device=seg_device)

# Cache SmallSegUNet masks once for all episodes
mask_cache_dir = pathlib.Path(results_path, "demonstration", f"{demopack_name}_masks")
mask_paths = cache_episode_masks(
    src_demo_root,
    mask_cache_dir,
    all_episodes,
    demonstration_cam,
    segmenter,
    overwrite=overwrite_dr_demos,
)
print(f"Cached {len(mask_paths)} episode mask archives in {mask_cache_dir}")

# Render all-techniques contact sheet on a held-out vp_testing frame
rep_ep = test_episodes[0]
_, rep_frames = read_video_rgb(src_demo_root / rep_ep / f"video_{demonstration_cam}.mp4")
rep_idx = len(rep_frames) // 2
with np.load(mask_paths[rep_ep]) as data:
    rep_mask = data["masks"][rep_idx]
contact_sheet_path = pathlib.Path(results_path, "dr_techniques_contact_sheet.png")
make_techniques_contact_sheet(rep_frames[rep_idx], rep_mask, RECIPES_DIR, contact_sheet_path)
display(Markdown(f"### Domain Randomization Techniques Contact Sheet (`{rep_ep}` frame {rep_idx})"))
display(Image(filename=str(contact_sheet_path)))

In [ ]:
demo_episode_maps = {demopack_name: {ep: [ep] for ep in all_episodes}}

def materialize_run(run_name, recipe_name, episodes, *, copies=1, src_run=None, use_live_seg=False):
    dr = DomainRandomizer.from_yaml(RECIPES_DIR / f"{recipe_name}.yaml")
    s_run = demopack_name if src_run is None else src_run
    s_yaml = pathlib.Path(Config().get_exprun_path(), "demonstration", f"{s_run}.yaml")
    s_root = pathlib.Path(Config().get_experiment("demonstration", s_run).data_dir())
    d_yaml = pathlib.Path(Config().get_exprun_path(), "demonstration", f"{run_name}.yaml")
    d_root = pathlib.Path(results_path, "demonstration", run_name)
    summary = materialize_randomized_demopack(
        src_yaml_path=s_yaml,
        dst_yaml_path=d_yaml,
        src_root=s_root,
        dst_root=d_root,
        dst_demo_run=run_name,
        episodes=episodes,
        camera=demonstration_cam,
        randomizer=dr,
        mask_paths=None if use_live_seg else mask_paths,
        live_segmenter=segmenter if use_live_seg else None,
        copies_per_episode=copies,
        preview_path=d_root / f"preview_{recipe_name}.png",
        overwrite=overwrite_dr_demos,
    )
    Config().get_experiment("demonstration", run_name, creation_style="exist-ok")
    demo_episode_maps[run_name] = summary["episode_map"]
    print(f"Materialized {run_name} ({recipe_name}): {summary['total_frames']} frames, cached={summary['cached_videos']}")
    return summary

# 1. Segmented white run (needed if seg_both or seg_train_real_test enabled)
seg_white_run = f"{demopack_name}_seg_white"
if "seg_both" in conditions or "seg_train_real_test" in conditions:
    materialize_run(seg_white_run, "seg_white", all_episodes, copies=1)

# 2. Training DR runs (randomized strictly on sp_training + vp_training; val/test stay real)
dr_recipe_runs = {}
for cond in conditions:
    if cond.startswith("dr_"):
        r_run = f"{demopack_name}_{cond}"
        materialize_run(r_run, cond, train_episodes, copies=int(copies_per_episode))
        dr_recipe_runs[cond] = r_run

# 3. Suite B1: Disjoint held-out OOD clutter + lighting stress test on vp_testing
ood_run = f"{demopack_name}_eval_ood_clutter"
ood_summary = materialize_run(ood_run, "eval_ood_clutter", test_episodes, copies=1)
display(Markdown("### Suite B1 Cluttered OOD Test Frames (`eval_ood_clutter`)"))
display(Image(filename=ood_summary["preview_path"]))

# 4. Honest live SmallSegUNet re-segmentation of Suite B1 cluttered frames for seg_both
ood_seg_run = f"{demopack_name}_eval_ood_clutter_seg_white"
if "seg_both" in conditions:
    ood_seg_summary = materialize_run(
        ood_seg_run, "seg_white", test_episodes, copies=1, src_run=ood_run, use_live_seg=True
    )
    display(Markdown("### Honest Live `SmallSegUNet` Re-Segmentation on Suite B1 Cluttered Frames"))
    display(Image(filename=ood_seg_summary["preview_path"]))

## Stage 1 & 2: Generate SP, VP, Eval-Only Alias, and Comparison Exp/Runs

In [ ]:
def make_split_entries(demo_run, episodes, camera):
    ep_map = demo_episode_maps.get(demo_run)
    entries = []
    for ep in episodes:
        expanded = ep_map[ep] if (ep_map is not None and ep in ep_map) else [ep]
        for dst_ep in expanded:
            entries.append([demo_run, dst_ep, camera])
    return entries


def write_exprun_yaml(exp_name, run_name, val):
    path = pathlib.Path(Config().get_exprun_path(), exp_name, f"{run_name}.yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(val, f)
    return path


def generate_sp_conv_vae_neo(params, exp_name, run_name):
    val = copy.copy(params)
    val["class"] = "ConvVaeNeoSensorProcessing"
    val["input-to-notebook"] = [
        "sensorprocessing/Train_Conv_VAE_Neo.ipynb",
        "sensorprocessing/Verify_Conv_VAE_Neo.ipynb",
    ]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_sp_propriotuned_vit(params, exp_name, run_name):
    val = copy.copy(params)
    val["output_size"] = 6
    val["batch_size"] = 32
    val["vit_model"] = "vit_b_16"
    val["vit_output_dim"] = 768
    val["projection_hidden_dim"] = 512
    val["vit_weights"] = "DEFAULT"
    val["proprio_step_1"] = 64
    val["proprio_step_2"] = 64
    val["input-to-notebook"] = ["sensorprocessing/Train_ProprioTuned_ViT.ipynb"]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_sp_propriotuned_cnn(params, exp_name, run_name):
    val = copy.copy(params)
    val["output_size"] = 6
    val["batch_size"] = 32
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_vp_train(params, exp_name, run_name):
    val = copy.copy(params)
    val["input-to-notebook"] = [
        "visual_proprioception/Train_VisualProprioception.ipynb",
        "visual_proprioception/Verify_VisualProprioception.ipynb",
    ]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_vp_compare(params, collection_experiment, run_name):
    val = copy.copy(params)
    val["name"] = run_name
    val["source_experiment"] = "visual_proprioception"
    val["input-to-notebook"] = ["visual_proprioception/Compare_VisualProprioception.ipynb"]
    write_exprun_yaml(collection_experiment, run_name, val)
    return flow_entry(f"Compare_{run_name}", collection_experiment, run_name, 0, compare_creation_style)

In [ ]:
# Map each evaluation condition to its underlying training regime, validation run, and held-out runs
CONDITION_SPECS = {
    "orig": {
        "regime": "orig",
        "train_demo": demopack_name,
        "val_demo": demopack_name,
        "clean_demo": demopack_name,
        "ood_demo": ood_run,
    },
    "seg_both": {
        "regime": "seg_white",
        "train_demo": seg_white_run,
        "val_demo": seg_white_run,
        "clean_demo": seg_white_run,
        "ood_demo": ood_seg_run,
    },
    "seg_train_real_test": {
        "regime": "seg_white",
        "train_demo": seg_white_run,
        "val_demo": seg_white_run,
        "clean_demo": demopack_name,
        "ood_demo": ood_run,
        "alias_of_condition": "seg_both",
    },
}
for cond, r_run in dr_recipe_runs.items():
    CONDITION_SPECS[cond] = {
        "regime": cond,
        "train_demo": r_run,
        "val_demo": demopack_name,
        "clean_demo": demopack_name,
        "ood_demo": ood_run,
    }

enabled_archs = []
if do_VAE_NEO:
    enabled_archs.append("vae_neo")
if do_VIT:
    enabled_archs.append("vit_base")
if do_VGG:
    enabled_archs.append("vgg19")
if do_RESNET:
    enabled_archs.append("resnet50")

train_stages = []
sp_registry = {}
vp_trained_registry = {}
eval_aliases = []
clean_compare_runs = []
ood_compare_runs = []

def ensure_sp(arch, latent_size, regime, train_demo, val_demo):
    key = (arch, latent_size, regime)
    if key in sp_registry:
        return sp_registry[key]
    sp_train = make_split_entries(train_demo, selection["sp_training"], demonstration_cam)
    sp_val = make_split_entries(val_demo, selection["sp_validation"], demonstration_cam)
    run_name = f"sp_{arch}_{latent_size}_{regime}"

    if arch == "vae_neo":
        exp_name = "sensorprocessing_conv_vae_neo"
        params = {
            "latent_size": latent_size,
            "image_size": list(vae_image_size),
            "frame_stride": int(vae_frame_stride),
            "epochs": int(epochs_sp_vae),
            "early_stopping_patience": max(10, int(epochs_sp_vae)),
            "training_data": sp_train,
            "validation_data": sp_val,
        }
        entry = generate_sp_conv_vae_neo(params, exp_name, run_name)
        sp_class = "ConvVaeNeoSensorProcessing"
    elif arch == "vit_base":
        exp_name = "sensorprocessing_propriotuned_Vit"
        params = {
            "image_size": [224, 224],
            "latent_size": latent_size,
            "epochs": int(epochs_sp_vit),
            "training_data": sp_train,
            "validation_data": sp_val,
            "class": "Vit",
            "model": "VitProprioTunedRegression",
            "freeze_feature_extractor": False,
            "loss": "MSELoss",
            "learning_rate": 0.0001,
        }
        entry = generate_sp_propriotuned_vit(params, exp_name, run_name)
        sp_class = "VitSensorProcessing"
    elif arch in ("vgg19", "resnet50"):
        exp_name = "sensorprocessing_propriotuned_cnn"
        params = {
            "image_size": list(image_size),
            "latent_size": latent_size,
            "epochs": int(epochs_sp_cnn),
            "training_data": sp_train,
            "validation_data": sp_val,
            "loss": "MSELoss",
            "learning_rate": 0.001,
        }
        if arch == "vgg19":
            params["class"] = "VGG19ProprioTunedSensorProcessing"
            params["model"] = "VGG19ProprioTunedRegression"
        else:
            params["class"] = "ResNetProprioTunedSensorProcessing"
            params["model"] = "ResNetProprioTunedRegression"
            params["freeze_feature_extractor"] = True
            params["reductor_step_1"] = 512
            params["proprio_step_1"] = 64
            params["proprio_step_2"] = 16
        entry = generate_sp_propriotuned_cnn(params, exp_name, run_name)
        sp_class = params["class"]
    else:
        raise ValueError(f"Unknown arch: {arch}")

    info = {"experiment": exp_name, "run": run_name, "sp_class": sp_class}
    sp_registry[key] = info
    train_stages.append(entry)
    return info


def make_vp_params(run_name, latent_size, sp_info, train_demo, val_demo, heldout_demo):
    return {
        "name": run_name,
        "output_size": 6,
        "encoding_size": latent_size,
        "training_data": make_split_entries(train_demo, selection["vp_training"], demonstration_cam),
        "validation_data": make_split_entries(val_demo, selection["vp_validation"], demonstration_cam),
        "heldout_data": make_split_entries(heldout_demo, selection["vp_testing"], demonstration_cam),
        "proprioception_heldout_input_file": "heldout_inputs.pt",
        "proprioception_heldout_target_file": "heldout_targets.pt",
        "regressor_hidden_size_1": 64,
        "regressor_hidden_size_2": 64,
        "loss": "MSE",
        "epochs": int(epochs_vp),
        "batch_size": 64,
        "sensor_processing": sp_info["sp_class"],
        "sp_experiment": sp_info["experiment"],
        "sp_run": sp_info["run"],
    }


for latent_size in latent_sizes:
    for arch in enabled_archs:
        for cond in conditions:
            spec = CONDITION_SPECS[cond]
            sp_info = ensure_sp(arch, latent_size, spec["regime"], spec["train_demo"], spec["val_demo"])
            vp_run_name = f"vp_{arch}_{latent_size}_{cond}"

            alias_of = spec.get("alias_of_condition")
            if alias_of is not None and (arch, latent_size, alias_of) in vp_trained_registry:
                src_vp_run = vp_trained_registry[(arch, latent_size, alias_of)]
                clean_params = make_vp_params(
                    vp_run_name, latent_size, sp_info, spec["train_demo"], spec["val_demo"], spec["clean_demo"]
                )
                write_exprun_yaml("visual_proprioception", vp_run_name, clean_params)
                eval_aliases.append((vp_run_name, src_vp_run))
            else:
                vp_params = make_vp_params(
                    vp_run_name, latent_size, sp_info, spec["train_demo"], spec["val_demo"], spec["clean_demo"]
                )
                train_stages.append(generate_vp_train(vp_params, "visual_proprioception", vp_run_name))
                vp_trained_registry[(arch, latent_size, cond)] = vp_run_name
                src_vp_run = vp_run_name

            clean_compare_runs.append(vp_run_name)

            # Eval-only OOD alias sharing exact trained VP weights, evaluated on Suite B1
            ood_vp_run = f"{vp_run_name}_ood"
            ood_params = make_vp_params(
                ood_vp_run, latent_size, sp_info, spec["train_demo"], spec["val_demo"], spec["ood_demo"]
            )
            write_exprun_yaml("visual_proprioception", ood_vp_run, ood_params)
            eval_aliases.append((ood_vp_run, src_vp_run))
            ood_compare_runs.append(ood_vp_run)

collection_experiment = "visual_proprioception_collections"
compare_stages = [
    generate_vp_compare(
        {"tocompare": clean_compare_runs, "comparison_dataset": "heldout_data"},
        collection_experiment,
        "vp_comp_clean",
    ),
    generate_vp_compare(
        {"tocompare": ood_compare_runs, "comparison_dataset": "heldout_data"},
        collection_experiment,
        "vp_comp_ood",
    ),
]
all_stages = train_stages + compare_stages
print(f"Scheduled {len(train_stages)} training stages, {len(eval_aliases)} eval-only aliases, and {len(compare_stages)} comparisons:")
for item in all_stages:
    print(f"  - {item['name']}: {item['experiment']}/{item['run']}")

## Stage 3: Run Training, Wire Eval-Only Aliases, Run Suite A & Suite B1 Comparisons

In [ ]:
def stage_already_trained(entry):
    if entry.get("creation_style") != "exist-ok":
        return False
    stage_dir = pathlib.Path(results_path, entry["experiment"], entry["run"])
    return stage_dir.is_dir() and any(stage_dir.glob("*.pth"))

train_to_run = [e for e in train_stages if not stage_already_trained(e)]
skipped = [e["name"] for e in train_stages if stage_already_trained(e)]
if skipped:
    print(f"Skipping {len(skipped)} already-trained stages: {skipped}")

flow_error = None
try:
    if train_to_run:
        run_flow(train_to_run, expruns_path, results_path, notebooks_path)
    # Copy trained VP weights into eval-only aliases (seg_train_real_test & Suite B1 *_ood)
    for alias_run, src_run in eval_aliases:
        src_pth = pathlib.Path(results_path, "visual_proprioception", src_run, "proprioception_mlp.pth")
        if not src_pth.is_file():
            raise FileNotFoundError(f"Expected trained VP weights for alias {alias_run}: {src_pth}")
        dst_dir = pathlib.Path(results_path, "visual_proprioception", alias_run)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst_pth = dst_dir / "proprioception_mlp.pth"
        if (not dst_pth.is_file()) or (src_pth.stat().st_mtime > dst_pth.stat().st_mtime):
            shutil.copy2(src_pth, dst_pth)
            for stale in ("heldout_inputs.pt", "heldout_targets.pt"):
                (dst_dir / stale).unlink(missing_ok=True)
    run_flow(compare_stages, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
clean_dir = pathlib.Path(results_path, "visual_proprioception_collections", "vp_comp_clean")
ood_dir = pathlib.Path(results_path, "visual_proprioception_collections", "vp_comp_ood")

clean_previews = [
    p for p in (clean_dir / "msecomparison23.png", clean_dir / "comparison23.png")
    if p.is_file()
]
report = display_flow_report(
    all_stages,
    results_path,
    notebooks_path,
    clean_dir,
    flow_error=flow_error,
    preview_files=clean_previews,
)

if ood_dir.is_dir():
    display(Markdown("### Suite B1: Cluttered OOD Held-Out (`vp_comp_ood`, `*23` layout)"))
    for fname in ("msecomparison23.png", "comparison23.png"):
        p = ood_dir / fname
        if p.is_file():
            display(Image(filename=str(p)))

clean_csv = clean_dir / "all_msecomparison_values.csv"
ood_csv = ood_dir / "all_msecomparison_values.csv"
if clean_csv.is_file():
    _, md_table = summarize_dr_comparisons(clean_csv, ood_csv if ood_csv.is_file() else None)
    display(Markdown(f"### Unified Suite A (Clean) vs. Suite B1 (Cluttered OOD) Summary\n\n{md_table}"))

if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")